# BimaGyan — LoRA Fine-Tuning on Llama 3.1 8B (Cloudflare Workers AI Compatible)

**Base Model:** `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`  
**Architecture match:** Cloudflare Workers AI `@cf/meta/llama-3.1-8b-instruct`  
**LoRA Rank:** $r = 8$ (strict Cloudflare requirement for multi-tenant inference)  
**Target:** Saves PEFT LoRA adapter (`adapter_model.safetensors` < 50MB + `adapter_config.json`) and pushes to Hugging Face.

In [ ]:
# ── STEP 1: Install Dependencies ──────────────────────────────────────────────
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'unsloth[kaggle-new]',
    'trl',
    'peft',
    'datasets',
    'huggingface_hub',
], check=True)
print('Dependencies installed.')

In [ ]:
# ── STEP 2: Configuration & Credentials ───────────────────────────────────────
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    HF_REPO  = secrets.get_secret('HF_REPO') or 'siddharth-codes/bimagyan-llama3-lora'
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    HF_REPO  = os.environ.get('HF_REPO', 'siddharth-codes/bimagyan-llama3-lora')

DATASET_PATH = '/kaggle/input/datasets/siddharthkumarmishra/bimagyan-training-data/train.jsonl'
if not os.path.exists(DATASET_PATH):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.jsonl' in files:
            DATASET_PATH = os.path.join(root, 'train.jsonl')
            break

print(f'Dataset path: {DATASET_PATH}')
print(f'Target HF repo: {HF_REPO}')

In [ ]:
# ── STEP 3: Load Model & Attach LoRA (Rank 8 for Cloudflare) ──────────────────
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Attach LoRA with rank 8 (strict limit for Cloudflare Workers AI)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print('LoRA attached: rank=8, alpha=16 (Cloudflare Workers AI compatible)')

In [ ]:
# ── STEP 4: Format Dataset with Llama-3 Prompt Template ───────────────────────
from datasets import load_dataset

llama3_prompt = """<|start_header_id|>system<|end_header_id|>

{system}<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}{input_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

def formatting_prompts_func(examples):
    systems = examples.get("system", [""] * len(examples["instruction"]))
    instructions = examples["instruction"]
    inputs = examples.get("input", [""] * len(instructions))
    outputs = examples["output"]
    texts = []
    for sys_prompt, inst, inp, out in zip(systems, instructions, inputs, outputs):
        inp_part = f"\n\nContext:\n{inp}" if inp and inp.strip() else ""
        sys_text = sys_prompt or "You are BimaGyan AI, an objective, consumer-first Indian insurance education assistant."
        text = llama3_prompt.format(system=sys_text, instruction=inst, input_text=inp_part, output=out)
        texts.append(text)
    return {"text": texts}

dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train'].map(formatting_prompts_func, batched=True)
val_ds   = split['test'].map(formatting_prompts_func, batched=True)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# ── STEP 5: Train ─────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
import inspect

sig = inspect.signature(TrainingArguments.__init__)
eval_arg_name = 'eval_strategy' if 'eval_strategy' in sig.parameters else 'evaluation_strategy'

train_kwargs = {
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'warmup_steps': 10,
    'max_steps': 300,
    'learning_rate': 2e-4,
    'fp16': not torch.cuda.is_bf16_supported(),
    'bf16': torch.cuda.is_bf16_supported(),
    'logging_steps': 20,
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'lr_scheduler_type': 'cosine',
    'seed': 3407,
    'output_dir': '/kaggle/working/bimagyan_llama3_checkpoints',
    'save_strategy': 'no',
    eval_arg_name: 'no',
}

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(**train_kwargs),
)

print('Starting fine-tuning...')
trainer.train()
print('Training complete!')

In [ ]:
# ── STEP 6: Save LoRA Adapter for Cloudflare Workers AI ───────────────────────
output_dir = '/kaggle/working/bimagyan_lora_cloudflare'
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Ensure model_type is explicitly set to 'llama' in adapter_config.json
import json
config_path = os.path.join(output_dir, 'adapter_config.json')
with open(config_path, 'r') as f:
    cfg = json.load(f)
cfg['model_type'] = 'llama'
with open(config_path, 'w') as f:
    json.dump(cfg, f, indent=2)

files = os.listdir(output_dir)
print(f'LoRA files ready for Cloudflare: {files}')

# Push to Hugging Face Hub so we can easily download or deploy
if HF_TOKEN:
    print(f'Pushing LoRA adapter to {HF_REPO}...')
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print('Successfully pushed to Hugging Face!')
else:
    print('Note: Add HF_TOKEN secret to push adapter automatically.')